# ⚡ Practical 11 — Statistical Models for Time Series
## Stationarity → ACF/PACF → SARIMA → Forecast evaluation


**By the end, you will:**
- explain and test stationarity (ADF);
- apply first and seasonal differences to a quarterly series;
- interpret ACF and PACF;
- build SARIMA forecasts without using future data;
- compare forecasts against a seasonal-naive baseline using MAE and NRMSE;
- interpret residual autocorrelation.

**Instructions:** Run the cells in order. Complete every `TODO`. Answer the questions in Markdown directly under each exercise. Work with the **original GWh levels** for evaluation — do not compare a differenced forecast to the original series.


In [ ]:
# %pip -q install pandas numpy matplotlib statsmodels openpyxl

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
pd.set_option("display.precision", 3)
print("Ready to analyse quarterly electricity data")

## 1. Data Detective

The chapter uses **quarterly electricity production from Australia**, recorded at the start of March, June, September and December.

**Task:** Complete loading and indexing. The time index must be chronologically sorted and regularly spaced. `QS-MAR` means a quarter beginning in March (then June, September, December).

### Task 1 — Read the file and inspect the time index

Fill the `TODO` lines, then check the first five observations, number of rows, date range and missing values.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/drmatson/energy_informatics/main/2025/data/Australia_electricty_data.xlsx"
raw = pd.read_excel(DATA_URL, index_col=False)

# TODO 1: rename "Unnamed: 0" to "datetime" and "Electricity [GWh]" to "electricity"
df = ...

# TODO 2: convert datetime and electricity to correct types
df["datetime"] = ...
df["electricity"] = ...

df = df.dropna(subset=["datetime", "electricity"])
df = df.sort_values("datetime").drop_duplicates(subset="datetime")

# TODO 3: use datetime index, select electricity and make quarterly frequency explicit
y = ...

print(y.head())
print("Observations:", len(y))
print("Period:", y.index.min(), "to", y.index.max())
print("Missing quarters:", y.isna().sum())
assert y.notna().all(), "Inspect missing quarters before modeling"
assert y.index.is_monotonic_increasing

In [ ]:
# We use the exact historical period illustrated in the chapter and lecture.
series = y.loc["1956":"1968"].copy()
assert len(series) >= 40, "The selected window should contain several years"
fig, ax = plt.subplots()
series.plot(ax=ax, title="Australian electricity production, 1956–1968")
ax.set_ylabel("Production (GWh)")
plt.show()

**Question 1 (write 2–3 sentences):** Describe the long-term trend, the annual seasonal pattern and whether seasonal fluctuations seem to grow with the level.

**Your answer:** ...

## 2. Is it stationary?

The Augmented Dickey–Fuller (ADF) test has the null hypothesis **H₀: the series has a unit root**. At 5% significance, a small p-value is evidence *against* this null. A large p-value is **not proof** of non-stationarity — it means we cannot reject the unit-root null.

In [ ]:
def adf_report(x, label):
    x = pd.Series(x).dropna()
    statistic, pvalue, usedlag, nobs, critical, *_ = adfuller(x, autolag="AIC")
    return {"series": label, "ADF statistic": statistic,
            "p-value": pvalue, "lags": usedlag,
            "decision @ 5%": "reject unit root" if pvalue < .05 else "cannot reject unit root"}

print(pd.DataFrame([adf_report(series, "Original")]).to_string(index=False))

### Task 2 — Interpret ADF

Use the provided function to test the original series, then write a one-sentence decision using `p < 0.05`.

In [ ]:
# TODO: get the p-value for the original series
p_original = ...
print("p-value:", p_original)
print("Decision:", "Reject H0" if p_original < 0.05 else "Cannot reject H0")

**Question 2:** What is the ADF null hypothesis? Does `p > 0.05` prove that the series has a unit root? Explain.

**Your answer:** ...

## 3. Differencing Challenge

For quarterly data, one year corresponds to **4 observations**.

- First difference: $y_t-y_{t-1}$
- Seasonal difference: $y_t-y_{t-4}$
- Both: $(1-B)(1-B^4)y_t$

The order of these two difference operators does not matter mathematically (apart from the positions of initial missing values).

### Task 3 — Transform the series

Complete each transformation. Compare plots and run ADF on every version.

In [ ]:
# TODO: compute the three transformations; do not change `series`
first_diff = ...
seasonal_diff = ...
both_diff = ...

transforms = {"Original": series,
              "First diff": first_diff,
              "Seasonal diff": seasonal_diff,
              "Seasonal + first": both_diff}

fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, values) in zip(axes.flat, transforms.items()):
    values.plot(ax=ax, title=name)
    ax.set_ylabel("GWh or GWh difference")
plt.tight_layout()
plt.show()

adf_table = pd.DataFrame([adf_report(v, k) for k, v in transforms.items()])
print(adf_table.to_string(index=False))

**Question 3:** Which transformation is the *simplest plausible* choice for this series? Give the ADF p-value and one visual argument. What is lost when applying `diff(4).diff()`?

**Your answer:** ...

## 4. ACF/PACF Detective

ACF measures total autocorrelation at each lag. PACF measures the correlation after accounting for the intermediate lags. Use them on the **transformed** series. The ACF of trending original data can be dominated by non-stationarity.

For quarterly data, look at **lags 4, 8 and 12** (whole years). ACF/PACF suggest candidate orders; they do not uniquely identify a SARIMA model.

### Task 4 — Draw ACF and PACF

Fill both plot commands. Exclude lag zero from ACF to focus on informative lags.

In [ ]:
signal = both_diff  # You can change to seasonal_diff to compare
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TODO: draw ACF and PACF with 12 lags
...
...
axes[0].set_title("ACF: transformed production")
axes[1].set_title("PACF: transformed production")
plt.tight_layout()
plt.show()

# TODO: compute lag-4 autocorrelation of the original series
lag4_corr = ...
print("Original lag-4 autocorrelation:", round(lag4_corr, 3))

**Question 4:** Identify **two lags** worth investigating. Does the transformed series still show seasonal autocorrelation? Suggest *one* small non-seasonal $(p,q)$ pair for ARIMA and justify it.

**Your answer:** ...

### Optional interactive exploration

In JupyterLab/Colab, use the dropdown to compare ACF and PACF. This cell is **optional**; if widgets are missing, the static plots above are sufficient. Widgets are not required for a static Jupyter Book page.

In [ ]:
try:
    from ipywidgets import interact, Dropdown, IntSlider
    def show_lags(transformation="Seasonal + first", n_lags=12):
        data = transforms[transformation]
        n_lags = min(n_lags, (len(data) - 1) // 2)
        fig, axs = plt.subplots(1, 2, figsize=(11, 3))
        plot_acf(data, lags=n_lags, zero=False, ax=axs[0])
        plot_pacf(data, lags=n_lags, method="ywm", ax=axs[1])
        axs[0].set_title(transformation + " | ACF")
        axs[1].set_title(transformation + " | PACF")
        plt.tight_layout()
        plt.show()
    interact(show_lags,
             transformation=Dropdown(options=list(transforms), value="Seasonal + first"),
             n_lags=IntSlider(value=12, min=4, max=16, step=4))
except ImportError:
    print("Optional widgets unavailable. Static ACF/PACF plots above are enough.")

## 5. Forecasting without leakage

Keep the **last 8 quarters (2 years)** unseen. Models and any parameter choices should use only the training sample. For a fair baseline, a **Seasonal Naive** forecast repeats the last four observed quarters (one year) into the future.

*Important:* We fit SARIMA directly to the original GWh series with differencing **inside the model**. Do not manually difference the whole dataset and then fit `d=1, D=1` again.

### Task 5 — Time-aware train/test split

Complete the chronological split and the seasonal-naive baseline. Check both values and timestamps.

In [ ]:
# TODO: reserve the final eight observations for testing
train = ...
test = ...

# TODO: repeat the final four training quarters for the 8-step test horizon
seasonal_naive = pd.Series(..., index=test.index, name="Seasonal naive")

print("Train:", train.index[0], "to", train.index[-1], "| n=", len(train))
print("Test:", test.index[0], "to", test.index[-1], "| n=", len(test))
assert len(test) == 8
assert train.index.max() < test.index.min()
assert seasonal_naive.index.equals(test.index)

ax = series.plot(label="Observed", figsize=(11, 4))
seasonal_naive.plot(ax=ax, label="Seasonal naive", linestyle="--")
ax.axvline(test.index[0], linestyle=":", color="grey")
ax.set_ylabel("GWh")
ax.legend()
plt.show()

**Question 5:** Why is `train_test_split(..., shuffle=True)` unsuitable for this task? Does the seasonal-naive baseline use any observations from the test period?

**Your answer:** ...

## 6. SARIMA competition

SARIMA notation: $(p,d,q)\times(P,D,Q)_4$. Fit **two small candidates** on train, compare training AIC, choose the lower-AIC model **before examining test errors** and forecast the entire 8-quarter holdout.

- **Model A:** SARIMA(0,1,1) × (0,1,1)₄
- **Model B:** SARIMA(1,1,0) × (1,1,0)₄

The seasonal period is `4`. AIC compares fitted candidate models on the same training data; a lower AIC does **not** guarantee a better out-of-sample forecast.

### Task 6 — Fit SARIMA and forecast

Fill the model constructor and the forecasting call. Model B is deliberately different from A. Inspect convergence and training AIC.

In [ ]:
candidate_orders = {
    "SARIMA A": ((0, 1, 1), (0, 1, 1, 4)),
    "SARIMA B": ((1, 1, 0), (1, 1, 0, 4)),
}

fits = {}
for name, (order, seasonal_order) in candidate_orders.items():
    # TODO: create SARIMAX on *train* with these orders
    model = ...
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        result = model.fit(disp=False, maxiter=200)
    fits[name] = result
    converged = result.mle_retvals.get("converged", True)
    print(f"{name} | AIC={result.aic:.1f} | converged={converged}")

# TODO: select lowest AIC (using the training fits only)
best_name = ...
best_fit = fits[best_name]

# TODO: forecast exactly len(test) quarters
forecast_result = ...
forecast = pd.Series(forecast_result.predicted_mean, index=test.index, name=best_name)
interval = forecast_result.conf_int(alpha=0.05)
print("Selected on train AIC:", best_name)
print(forecast.head())

In [ ]:
# Plot GWh levels (not differenced values)
fig, ax = plt.subplots(figsize=(11, 4))
train.iloc[-16:].plot(ax=ax, label="Train (last 4 years)")
test.plot(ax=ax, marker="o", label="Test actual")
seasonal_naive.plot(ax=ax, linestyle="--", label="Seasonal naive")
forecast.plot(ax=ax, marker="o", label=best_name)
ax.fill_between(test.index, interval.iloc[:, 0].to_numpy(),
                interval.iloc[:, 1].to_numpy(), alpha=0.15,
                label="SARIMA 95% prediction interval")
ax.set_ylabel("Electricity production (GWh)")
ax.legend()
plt.show()

## 7. Who forecasts better?

Use **MAE** and **NRMSE**. Here we explicitly define:

$\mathrm{NRMSE} (\%) = 100 \times \frac{\sqrt{\mathrm{mean}((y-\hat y)^2)}}{\mathrm{mean}(y_{test})}$

This normalization uses the *mean observed test value for reporting*. Since electricity production is positive, the denominator is nonzero. A lower error is better.

### Task 7 — Compute the two metrics

Finish the metric function and compare the seasonal-naive forecast with SARIMA. Do not rank the models using training AIC here.

In [ ]:
def evaluate(truth, prediction):
    # TODO: use NumPy; check indices in the cell below
    mae = ...
    rmse = ...
    nrmse_pct = ...
    return {"MAE (GWh)": mae, "NRMSE (%)": nrmse_pct}

assert test.index.equals(seasonal_naive.index)
assert test.index.equals(forecast.index)
scoreboard = pd.DataFrame({
    "Seasonal naive": evaluate(test, seasonal_naive),
    best_name: evaluate(test, forecast)
}).T
print(scoreboard.round(2).to_string())
print("Lowest test NRMSE:", scoreboard["NRMSE (%)"].idxmin())

**Question 6:** Which method has the smaller **test NRMSE**? Is it necessarily the one with the smaller AIC? How large is the improvement (or deterioration) in percentage points?

**Your answer:** ...

## 8. Residuals: is there still a pattern?

A satisfactory residual series should show no important *remaining autocorrelation*. A p-value above 0.05 in the Ljung–Box test means **insufficient evidence against residual independence at the tested lags**, not proof of perfect white noise.

Do not confuse **model residuals** with **test forecast errors**: residual diagnostics here are evaluated on training data.

### Task 8 — Diagnose the fitted model

Plot training residuals and their ACF. Use Ljung–Box at lag 8; skip initialization observations which can distort the diagnostic.

In [ ]:
burn = max(6, best_fit.loglikelihood_burn)
residuals = best_fit.resid.iloc[burn:].dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
residuals.plot(ax=axes[0], title="Training model residuals")
axes[0].axhline(0, color="grey", linestyle="--")
# TODO: ACF of training residuals, 8 lags
...
plt.tight_layout()
plt.show()

# TODO: Ljung–Box test with lag 8 (returns a DataFrame)
lb = ...
print(lb)

**Question 7:** Are the residuals approximately structureless? Mention the ACF and Ljung–Box p-value. Would you deploy the model based on this 8-quarter test alone? Why or why not?

**Your answer:** ...



Try **one** more low-order SARIMA candidate, e.g. $(1,1,1)\times(0,1,1)_4$. Compare training AIC and residual diagnostics, *then* evaluate the test forecast only once you have made a model choice. Avoid tuning repeatedly on this single test period: that is another form of leakage.

In [ ]:
# OPTIONAL EXPERIMENT: run only after explaining your original choice.
# order = (1, 1, 1)
# seasonal_order = (0, 1, 1, 4)
# trial = SARIMAX(train, order=order, seasonal_order=seasonal_order,
#                 trend="n", enforce_stationarity=False,
#                 enforce_invertibility=False).fit(disp=False)
# print("Trial train AIC:", trial.aic)